# CPU/GPU/TPU profiling analysis

This notebook analyzes the controlled full-861-decision hardware matrix. It does not run models. Copy the completed hardware CSV into `results/hardware_comparison.csv` after the authorized compute task finishes.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
path = ROOT / 'results' / 'hardware_comparison.csv'
if not path.exists():
    raise FileNotFoundError('Complete the authorized full-run matrix and add results/hardware_comparison.csv')
df = pd.read_csv(path)
assert set(df['n']) == {861}, 'The ME344 result must use all 861 decisions.'
df

In [ ]:
required = ['hardware', 'concurrency', 'decisions_per_second', 'latency_p50_ms', 'latency_p95_ms', 'peak_memory_gb', 'status']
missing = [c for c in required if c not in df.columns]
assert not missing, f'Missing columns: {missing}'
complete = df[df['status'].eq('complete')].copy()
complete.sort_values(['concurrency', 'decisions_per_second'], ascending=[True, False])

In [ ]:
ax = (complete[complete['concurrency'].eq(24)]
      .sort_values('decisions_per_second')
      .plot.barh(x='hardware', y='decisions_per_second', legend=False, color='#4285F4'))
ax.set(title='Grading optimizes for throughput', xlabel='Decisions per second', ylabel='')
plt.tight_layout()

## Interpretation contract

Compare hardware only when checkpoint, prompt, parser, output limit, batching, and concurrency match. Report unavailable telemetry as unavailable. The earlier 27B result came from an unbatched endpoint and must not be interpreted as an A100 hardware ceiling.